# CT analysis of object 1 

22481 Introduction to Medical Imaging - CT analysis of object 1 

Helical CT series (520 slices, pixel spacing 0.586 mm, slice spacing 0.625 mm).

1) Radial HU profile on the mean of 13 slices (Locations 117-125 mm).
2) Objective length and volume: slice-by-slice tracking, 3-sigma detection against
   plain gel, diameter from the radial profile, Cavalieri volume, sensitivity analysis.

DICOM reading functions are adapted from the course Image Analysis Toolbox.
Requires: numpy, scipy, matplotlib, pydicom.

Run the cells in order. Only edit the **Settings** cell (`DATA_PATH` and, if needed, the coordinates).

In [ ]:
%matplotlib inline

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pydicom
from scipy import ndimage


## Settings

In [ ]:
DATA_PATH = Path(r"DTU 2026 CT/DTU 2026 CT/Dtu2_DTU2/20260922/Helical")   # Helical series folder
W, L = 50, 15                    # display window / level (HU)
CENTRE_XY = (335, 316)           # centre of object 1 (column, row), read in Weasis
GEL_XY = (350, 296)              # gel next to object 1 (column, row)
GEL_ROI_LOC = 120.5              # Location (mm) of the slice used for the gel ROI
PROFILE_LOCS = (117.0, 125.0)    # slices averaged for the radial profile
PROFILE_RMAX = 7.5               # mm, stays inside the gel block
START_LOC = 123.25               # starting slice for the tracking (object best seen)
CONTROL_XY = (300, 300)          # point in plain gel used to measure the noise
Z_MIN, Z_MAX = 85.0, 134.0       # tracking range: after the object 2, before the block edge


## DICOM reading (adapted from the course toolbox)

In [ ]:
def _read_dicom(path):
    """Read one DICOM image file; fail clearly if it has no pixel data."""
    ds = pydicom.dcmread(str(path))
    if not hasattr(ds, "PixelData"):
        raise ValueError(f"No image pixel data in {path}")
    return ds

def _pixels(ds):
    """Return rescaled pixel values as floats (HU for CT if the rescale is correct)."""
    a = np.asarray(ds.pixel_array).astype(float)
    slope = float(getattr(ds, "RescaleSlope", 1))
    intercept = float(getattr(ds, "RescaleIntercept", 0))
    return a * slope + intercept

def load_dicom(path, series_uid=None):
    """Load a DICOM file or a single series from a directory (teacher's toolbox)."""
    path = Path(path)
    if path.is_file():
        ds = _read_dicom(path)
        return _pixels(ds), ds, [ds]
    if not path.is_dir():
        raise FileNotFoundError(path)
    groups = {}
    for p in sorted(path.rglob("*")):
        if not p.is_file() or p.name.startswith("."):
            continue
        try:
            h = pydicom.dcmread(str(p), stop_before_pixels=True)
            if not hasattr(h, "Rows"):
                continue
            uid = str(getattr(h, "SeriesInstanceUID", "unknown"))
            groups.setdefault(uid, []).append(p)
        except Exception:
            continue
    if not groups:
        raise ValueError("No readable DICOM image files found")
    print("Available series:")
    for uid, paths in groups.items():
        print(f"  {uid} : {len(paths)} files")
    if series_uid is None and len(groups) > 1:
        raise ValueError("Multiple series: set SERIES_UID to one listed above")
    uid = series_uid or next(iter(groups))
    if uid not in groups:
        raise ValueError("SERIES_UID not found")
    dsets = [_read_dicom(p) for p in groups[uid]]
    def sort_key(ds):
        try:
            orient = np.asarray(ds.ImageOrientationPatient, float)
            normal = np.cross(orient[:3], orient[3:])
            return float(np.dot(np.asarray(ds.ImagePositionPatient, float), normal))
        except Exception:
            return float(getattr(ds, "InstanceNumber", 0))
    dsets.sort(key=sort_key)
    arrs = [_pixels(ds) for ds in dsets]
    if len({a.shape for a in arrs}) != 1:
        raise ValueError("Different image shapes within series")
    return np.stack(arrs, axis=0), dsets[0], dsets

def pixel_spacing_mm(ds):
    """Return (row_spacing, column_spacing) in mm/pixel, or None if absent."""
    if hasattr(ds, "PixelSpacing"):
        return tuple(float(v) for v in ds.PixelSpacing)
    return None

def slice_locations(slices):
    """Location of each slice in mm (the same number Weasis shows as 'Location')."""
    locs = []
    for s in slices:
        if hasattr(s, "SliceLocation"):
            locs.append(float(s.SliceLocation))
        else:
            locs.append(float(s.ImagePositionPatient[2]))
    return np.array(locs)


## Figure style and basic measurements

In [ ]:
def style_axes(ax, title, xlabel, ylabel):
    """Same formatting as Luna's figures (title 18, axis titles 16, axis values 14, all bold)."""
    ax.set_title(title, fontsize=18, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=16, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=16, fontweight="bold")
    ax.tick_params(axis="both", labelsize=14)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")

def wl_label(ax, W, L):
    """Window/Level text: size 15, white on black."""
    ax.text(0.02, 0.98, f"Window/Level: {W}/{L}", transform=ax.transAxes,
            color="white", fontsize=15, va="top", bbox=dict(facecolor="black", alpha=0.7))

def circle_stats(img, cx, cy, r_mm, spacing, r_inner_mm=0.0):
    """Mean, SD and number of pixels inside a circle (or a ring if r_inner_mm > 0).

    cx, cy: centre in pixels (column, row); radii in mm.
    """
    dy, dx = spacing
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    d = np.hypot((xx - cx) * dx, (yy - cy) * dy)
    m = (d <= r_mm) & (d >= r_inner_mm)
    v = img[m]
    return {"n_pixels": int(m.sum()), "mean": float(v.mean()), "sd": float(v.std(ddof=1))}

def radial_profile(volume, ks, centres, spacing, r_max_mm, step_mm=0.5):
    """Mean HU as a function of distance from the centre of a round object.

    Averages all pixels at the same distance (all directions) and over the
    slices in ks, each with its own centre (column, row). Much less noisy than
    a single line profile. Returns (radius_mm, mean_HU, standard_error, n_pixels).
    """
    dy, dx = spacing
    edges = np.arange(0, r_max_mm + step_mm, step_mm)
    sums = np.zeros(len(edges) - 1); sq = np.zeros_like(sums); n = np.zeros_like(sums)
    for k, (cx, cy) in zip(ks, centres):
        img = volume[k]
        yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
        d = np.hypot((xx - cx) * dx, (yy - cy) * dy)
        sel = d < r_max_mm
        b = np.digitize(d[sel], edges) - 1
        v = img[sel]
        sums += np.bincount(b, v, len(sums))[:len(sums)]
        sq += np.bincount(b, v**2, len(sums))[:len(sums)]
        n += np.bincount(b, None, len(sums))[:len(sums)]
    mean = sums / n
    sd = np.sqrt(np.maximum(sq / n - mean**2, 0))
    r = (edges[:-1] + edges[1:]) / 2
    return r, mean, sd / np.sqrt(n), n.astype(int)


## Tracking of object 1 along its length

In [ ]:
def _disk_ring_kernel(r_in_mm, r_out_mm, spacing):
    dy, dx = spacing
    R = int(np.ceil(r_out_mm / min(dx, dy)))
    yy, xx = np.mgrid[-R:R+1, -R:R+1]
    d = np.hypot(xx * dx, yy * dy)
    k = ((d >= r_in_mm) & (d <= r_out_mm)).astype(float)
    return k

def _nan_mean_filter(img, kernel):
    """Mean of the pixels under 'kernel', ignoring NaN (air) pixels."""
    valid = np.isfinite(img).astype(float)
    filled = np.where(np.isfinite(img), img, 0.0)
    s = ndimage.convolve(filled, kernel, mode="nearest")
    n = ndimage.convolve(valid, kernel, mode="nearest")
    with np.errstate(invalid="ignore", divide="ignore"):
        return s / n

def track_object(volume, loc, start_loc, start_xy, spacing, z_min, z_max, half_slab=4,
                 max_step_px=3, r_ring=(2.5, 5.5), r_gel=(6.5, 8.5), air_hu=-200):
    """Follow a dark ring-shaped object slice by slice, up and down from start_loc.

    For every slice, the mean of 2*half_slab+1 neighbouring slices is used (less noise).
    'score' = mean HU in a gel ring around the object - mean HU in the object's dark ring
    (positive when the dark ring is there). The centre may move by at most max_step_px
    pixels from one slice to the next. Air pixels (< air_hu) are ignored.
    """
    k_ring = _disk_ring_kernel(*r_ring, spacing)
    k_gel = _disk_ring_kernel(*r_gel, spacing)
    pad = k_gel.shape[0] // 2 + max_step_px + 2
    k0 = int(np.argmin(np.abs(loc - start_loc)))
    order = np.argsort(loc)                      # slice indices sorted by Location
    pos = int(np.where(order == k0)[0][0])
    res = {}
    for direction in (+1, -1):
        cx, cy = start_xy
        p = pos if direction == +1 else pos - 1
        while 0 <= p < len(order):
            k = order[p]
            if not (z_min <= loc[k] <= z_max):
                break
            lo, hi = max(k - half_slab, 0), min(k + half_slab + 1, volume.shape[0])
            x0, x1 = int(cx) - pad, int(cx) + pad + 1
            y0, y1 = int(cy) - pad, int(cy) + pad + 1
            crop = volume[lo:hi, y0:y1, x0:x1].mean(axis=0)
            crop = np.where(crop < air_hu, np.nan, crop)
            score = _nan_mean_filter(crop, k_gel) - _nan_mean_filter(crop, k_ring)
            c = pad                               # previous centre inside the crop
            win = score[c-max_step_px:c+max_step_px+1, c-max_step_px:c+max_step_px+1]
            if np.all(~np.isfinite(win)):
                break
            j, i = np.unravel_index(np.nanargmax(win), win.shape)
            cx, cy = cx + (i - max_step_px), cy + (j - max_step_px)
            res[k] = (float(loc[k]), cx, cy, float(np.nanmax(win)))
            p += direction
    ks = sorted(res, key=lambda k: res[k][0])
    return {"k": np.array(ks),
            "location": np.array([res[k][0] for k in ks]),
            "cx": np.array([res[k][1] for k in ks], float),
            "cy": np.array([res[k][2] for k in ks], float),
            "score": np.array([res[k][3] for k in ks])}

def radial_profile_nan(volume, ks, centres, spacing, r_max_mm, step_mm=0.5, air_hu=-200):
    """Like radial_profile, but air pixels (< air_hu) are ignored (object near the block edge)."""
    dy, dx = spacing
    edges = np.arange(0, r_max_mm + step_mm, step_mm)
    nb = len(edges) - 1
    s = np.zeros(nb); sq = np.zeros(nb); n = np.zeros(nb)
    for k, (cx, cy) in zip(ks, centres):
        R = int(np.ceil(r_max_mm / min(dx, dy))) + 1
        x0, y0 = int(round(cx)) - R, int(round(cy)) - R
        img = volume[k, y0:y0 + 2*R + 1, x0:x0 + 2*R + 1]
        yy, xx = np.mgrid[y0:y0 + 2*R + 1, x0:x0 + 2*R + 1]
        d = np.hypot((xx - cx) * dx, (yy - cy) * dy)
        sel = (d < r_max_mm) & (img > air_hu)
        b = np.digitize(d[sel], edges) - 1
        v = img[sel]
        s += np.bincount(b, v, nb)[:nb]; sq += np.bincount(b, v**2, nb)[:nb]; n += np.bincount(b, None, nb)[:nb]
    with np.errstate(invalid="ignore", divide="ignore"):
        m = s / n
        se = np.sqrt(np.maximum(sq / n - m**2, 0)) / np.sqrt(n)
    return (edges[:-1] + edges[1:]) / 2, m, se, n.astype(int)

def analyse_along_length(volume, loc, track, spacing, present, n_per_window=9,
                         r_max_mm=8.5, r_core=2.0, r_ring=(2.5, 5.0), r_gel=(6.5, 8.5)):
    """Windows of n_per_window slices along the object (only where 'present').

    Centres come from a straight-line fit of the tracked centres (the object is straight).
    For each window: core, ring and gel HU (mean +/- standard error), edge radius (half-way
    between ring minimum and gel) and cross-sectional area. Volume = sum(area * thickness).
    """
    dz = float(np.median(np.abs(np.diff(np.sort(loc)))))
    z = track["location"]
    w = np.clip(track["score"], 0, None) * present
    fx = np.polyfit(z[present], track["cx"][present], 1, w=w[present])
    fy = np.polyfit(z[present], track["cy"][present], 1, w=w[present])
    ks = track["k"][present]; zs = z[present]
    out = {k: [] for k in ["location", "core", "core_se", "ring", "ring_se", "gel", "gel_se",
                           "r_edge", "area", "n_slices"]}
    for i0 in range(0, len(ks), n_per_window):
        kk, zz = ks[i0:i0 + n_per_window], zs[i0:i0 + n_per_window]
        cents = [(np.polyval(fx, zi), np.polyval(fy, zi)) for zi in zz]
        r, m, se, n = radial_profile_nan(volume, kk, cents, spacing, r_max_mm)
        def band(a, b):
            sel = (r >= a) & (r <= b) & np.isfinite(m)
            mean = np.average(m[sel], weights=n[sel])
            err = np.sqrt(np.sum((n[sel] * se[sel])**2)) / n[sel].sum()
            return mean, err
        core, core_se = band(0, r_core)
        ring, ring_se = band(*r_ring)
        gel, gel_se = band(*r_gel)
        inner = r <= 6.0
        i_min = int(np.nanargmin(np.where(inner, m, np.nan)))
        half = (m[i_min] + gel) / 2
        r_edge = np.nan
        for j in range(i_min, len(r) - 1):
            if m[j] < half <= m[j + 1]:
                r_edge = r[j] + (half - m[j]) / (m[j + 1] - m[j]) * (r[j + 1] - r[j])
                break
        vals = [float(np.mean(zz)), core, core_se, ring, ring_se, gel, gel_se, r_edge,
                np.pi * r_edge**2 if np.isfinite(r_edge) else np.nan, len(kk)]
        for key, v in zip(out, vals):
            out[key].append(v)
    out = {k: np.array(v) for k, v in out.items()}
    ok = np.isfinite(out["area"])
    out["volume_mm3"] = float(np.sum(out["area"][ok] * out["n_slices"][ok] * dz))
    # length along the axis, from the fitted straight line
    z0, z1 = zs.min(), zs.max() + dz
    dxl = (np.polyval(fx, z1) - np.polyval(fx, z0)) * spacing[1]
    dyl = (np.polyval(fy, z1) - np.polyval(fy, z0)) * spacing[0]
    out["length_z_mm"] = float(z1 - z0)
    out["length_axis_mm"] = float(np.sqrt((z1 - z0)**2 + dxl**2 + dyl**2))
    out["tilt_deg"] = float(np.degrees(np.arctan2(np.hypot(dxl, dyl), z1 - z0)))
    out["fit"] = (fx, fy)
    return out


## Objective length and volume (3-sigma detection against plain gel)

In [ ]:
def control_score(volume, loc, control_xy, spacing, z_min, z_max, half_slab=4, max_step_px=3,
                  r_ring=(2.5, 5.5), r_gel=(6.5, 8.5), air_hu=-200):
    """Same 'score' as track_object, but at a FIXED point in plain gel (no object).
    Taking the max over the same small search window gives the score that noise alone produces."""
    k_ring = _disk_ring_kernel(*r_ring, spacing)
    k_gel = _disk_ring_kernel(*r_gel, spacing)
    pad = k_gel.shape[0] // 2 + max_step_px + 2
    cx, cy = control_xy
    out_z, out_s = [], []
    for k in np.argsort(loc):
        if not (z_min <= loc[k] <= z_max):
            continue
        lo, hi = max(k - half_slab, 0), min(k + half_slab + 1, volume.shape[0])
        crop = volume[lo:hi, cy-pad:cy+pad+1, cx-pad:cx+pad+1].mean(axis=0)
        crop = np.where(crop < air_hu, np.nan, crop)
        score = _nan_mean_filter(crop, k_gel) - _nan_mean_filter(crop, k_ring)
        win = score[pad-max_step_px:pad+max_step_px+1, pad-max_step_px:pad+max_step_px+1]
        out_z.append(loc[k]); out_s.append(np.nanmax(win))
    return np.array(out_z), np.array(out_s)

def contiguous_around(mask, z, z_start):
    """Keep only the continuous run of True values that contains z_start."""
    i0 = int(np.argmin(np.abs(z - z_start)))
    keep = np.zeros_like(mask)
    if not mask[i0]:
        return keep
    a = i0
    while a - 1 >= 0 and mask[a - 1]:
        a -= 1
    b = i0
    while b + 1 < len(mask) and mask[b + 1]:
        b += 1
    keep[a:b + 1] = True
    return keep

def objective_length_volume(volume, loc, spacing, start_loc, start_xy, control_xy, z_min, z_max,
                            k_sigma=3.0, half_slab=4):
    """Object present where its score > mean + k_sigma * SD of the control (gel-only) score.
    Volume = sum over slices of the cross-section area, corrected for the tilt.
    Diameters come from the windows that lie fully inside the object (reliable);
    the ends are treated in two ways: as a cylinder (upper value) or as a cone to zero (lower value)."""
    dz = float(np.median(np.abs(np.diff(np.sort(loc)))))
    trk = track_object(volume, loc, start_loc, start_xy, spacing, z_min, z_max, half_slab=half_slab)
    cz, cs = control_score(volume, loc, control_xy, spacing, z_min, z_max, half_slab=half_slab)
    thr = float(np.mean(cs) + k_sigma * np.std(cs, ddof=1))
    present = contiguous_around(trk["score"] > thr, trk["location"], start_loc)
    n_win = 2 * half_slab + 1
    res = {"threshold": thr, "noise_mean": float(np.mean(cs)), "noise_sd": float(np.std(cs, ddof=1)),
           "track": trk, "present": present, "control": (cz, cs)}
    if present.sum() < n_win:
        res.update(length_axis_mm=np.nan, tilt_deg=np.nan, V_low=np.nan, V_high=np.nan, along=None)
        return res
    along = analyse_along_length(volume, loc, trk, spacing, present, n_per_window=n_win)
    full = (along["n_slices"] == n_win) & np.isfinite(along["r_edge"])
    if full.sum() > 0:
        med = np.median(2 * along["r_edge"][full])
        full &= (2 * along["r_edge"] > 0.5 * med)   # discard windows where the edge was not found
    if full.sum() == 0:
        res.update(length_axis_mm=along["length_axis_mm"], tilt_deg=along["tilt_deg"],
                   V_low=np.nan, V_high=np.nan, along=along)
        return res
    zc, dc = along["location"][full], 2 * along["r_edge"][full]
    zs = trk["location"][present]
    d_high = np.interp(zs, zc, dc)                       # ends: same diameter as nearest window (cylinder)
    zlo, zhi = zs.min() - dz / 2, zs.max() + dz / 2
    d_low = d_high.copy()                                # ends: linear to zero at the end (cone)
    m1 = zs < zc.min(); m2 = zs > zc.max()
    d_low[m1] = dc[0] * (zs[m1] - zlo) / (zc.min() - zlo)
    d_low[m2] = dc[-1] * (zhi - zs[m2]) / (zhi - zc.max())
    corr = 1 / np.cos(np.radians(along["tilt_deg"]))     # axial section of a tilted rod is an ellipse
    res.update(length_axis_mm=along["length_axis_mm"], tilt_deg=along["tilt_deg"], along=along,
               d_reliable=(zc, dc),
               V_low=float(np.sum(np.pi * (d_low / 2)**2) * dz * corr),
               V_high=float(np.sum(np.pi * (d_high / 2)**2) * dz * corr))
    return res


def selected_slices(loc, loc_start, loc_end):
    lo, hi = min(loc_start, loc_end), max(loc_start, loc_end)
    return [k for k in range(len(loc)) if lo <= loc[k] <= hi]


## Load the Helical series

In [ ]:
volume, ds, slices = load_dicom(DATA_PATH)
loc = slice_locations(slices)
spacing = pixel_spacing_mm(ds)
print(f"Loaded {volume.shape[0]} slices, pixel spacing {spacing} mm")


## 1) Radial HU profile on the mean of 13 slices

In [ ]:
# ---------- 1) Radial HU profile on the mean of 13 slices ----------
k_gel = int(np.argmin(np.abs(loc - GEL_ROI_LOC)))
gel = circle_stats(volume[k_gel], *GEL_XY, 2.0, spacing)
print(f"Gel ROI (r = 2 mm): {gel['mean']:.1f} +/- {gel['sd']:.1f} HU, {gel['n_pixels']} pixels")

ks = selected_slices(loc, *PROFILE_LOCS)
slab = volume[ks].mean(axis=0)
r, m, se, npx = radial_profile(volume, ks, [CENTRE_XY] * len(ks), spacing,
                               r_max_mm=PROFILE_RMAX, step_mm=0.5)
cx, cy = CENTRE_XY
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax, im, t in [(axes[0], volume[ks[len(ks)//2]], f"One slice ({loc[ks[len(ks)//2]]:.1f} mm)"),
                  (axes[1], slab, f"Mean of {len(ks)} slices ({PROFILE_LOCS[0]:.0f}-{PROFILE_LOCS[1]:.0f} mm)")]:
    ax.imshow(im, cmap="gray", vmin=L - W/2, vmax=L + W/2)
    ax.set_xlim(cx - 35, cx + 37); ax.set_ylim(cy + 17, cy - 31)
    ax.plot(cx, cy, "y+", markersize=14, mew=2)
    wl_label(ax, W, L)
    style_axes(ax, t, "Column (pixels)", "Row (pixels)")
ax = axes[2]
ax.plot(r, m, "o-", color="tab:orange", linewidth=2, markersize=4)
ax.fill_between(r, m - se, m + se, color="tab:orange", alpha=0.25)
ax.axhline(gel["mean"], color="gray", ls="--", label="gel")
ax.legend(fontsize=12); ax.grid(alpha=0.3)
style_axes(ax, "Radial profile (mean over slices)", "Distance from centre (mm)", "Mean HU")
plt.tight_layout(); plt.show()
for ri, mi, si, ni in zip(r, m, se, npx):
    print(f"r = {ri:4.2f} mm: {mi:6.1f} +/- {si:4.1f} HU  ({ni} pixels)")


## 2) Objective length and volume (3-sigma detection)

In [ ]:
# ---------- 2) Objective length and volume ----------
main = objective_length_volume(volume, loc, spacing, START_LOC, CENTRE_XY, CONTROL_XY,
                               Z_MIN, Z_MAX, k_sigma=3.0, half_slab=4)
trk, pres = main["track"], main["present"]
cz, cs = main["control"]
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
ax = axes[0]
ax.plot(trk["location"], trk["score"], color="tab:orange", linewidth=2, label="Object 1")
ax.plot(cz, cs, color="gray", linewidth=1.5, label="plain gel (control)")
for k, ls in [(2, ":"), (3, "--"), (5, "-.")]:
    ax.axhline(main["noise_mean"] + k * main["noise_sd"], color="k", ls=ls, linewidth=1, label=f"{k}sigma")
ax.fill_between(trk["location"], ax.get_ylim()[0], ax.get_ylim()[1], where=pres,
                color="tab:orange", alpha=0.12, label="present (3sigma)")
ax.legend(fontsize=11); ax.grid(alpha=0.3)
style_axes(ax, "Contrast of object 1 vs plain gel", "Location (mm)", "Contrast score (HU)")
ax = axes[1]
if main["along"] is not None:
    al = main["along"]
    ax.plot(al["location"], 2 * al["r_edge"], "o", color="lightgray", label="all windows")
    zc, dc = main["d_reliable"]
    ax.plot(zc, dc, "o-", color="tab:orange", linewidth=2, label="reliable windows")
    ax.legend(fontsize=11)
ax.grid(alpha=0.3)
style_axes(ax, "Diameter along object 1", "Location (mm)", "Diameter (mm)")
plt.tight_layout(); plt.show()

print(f"Noise of the score in plain gel: mean {main['noise_mean']:.2f} HU, SD {main['noise_sd']:.2f} HU")
print(f"3sigma threshold: {main['threshold']:.2f} HU")
if main["along"] is not None:
    z_p = trk["location"][pres]
    print(f"Object present from {z_p.min():.1f} to {z_p.max():.1f} mm, tilt {main['tilt_deg']:.0f} deg")
    print(f"Length along the axis: {main['length_axis_mm']:.1f} mm")
    print(f"Reliable diameters: {', '.join(f'{d:.1f}' for d in main['d_reliable'][1])} mm")
    print(f"Volume: {main['V_low']/1000:.2f} cm^3 (ends as cones) to "
          f"{main['V_high']/1000:.2f} cm^3 (ends as cylinders)")


## 3) Sensitivity analysis

In [ ]:
# ---------- 3) Sensitivity analysis ----------
print("\nSensitivity analysis")
print(" threshold | averaged slices | length (mm) | volume (cm^3)")
rows = []
for k_sigma in (2.0, 3.0, 5.0):
    for half in (3, 4, 7):
        res = objective_length_volume(volume, loc, spacing, START_LOC, CENTRE_XY, CONTROL_XY,
                                      Z_MIN, Z_MAX, k_sigma=k_sigma, half_slab=half)
        rows.append((res["length_axis_mm"], res["V_low"], res["V_high"]))
        print(f"   {k_sigma:.0f}sigma      |       {2*half+1:2d}        |    {res['length_axis_mm']:5.1f}    | "
              f"{res['V_low']/1000:.2f} - {res['V_high']/1000:.2f}")
Ls = np.array([x[0] for x in rows]); Vs = np.array([x[1:] for x in rows]).ravel()
print(f"\nRange over all choices: length {np.nanmin(Ls):.1f}-{np.nanmax(Ls):.1f} mm, "
      f"volume {np.nanmin(Vs)/1000:.2f}-{np.nanmax(Vs)/1000:.2f} cm^3")
